##### Da wir uns dazu entschieden haben einen Sentence Transformer zu verwenden, behalten wir dieses Notebook als Baseline-Modell bei. Im ersten Schritt wandeln wir Text in Features um. Modelle können keinen Text verstehen, deshalb brauchen wir numerische Features. Hierzu nutzen wir die TF-IDF Methode. Diese gilt als Standard für Sentiment.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

#Wir nutzen hier das bereinigte Datenset
df= pd.read_csv("../data/BMW/clean_reviews.csv")

In [ ]:
df.head()

In [ ]:
df.isna().sum()

##### Einige Reviews enthalten keine Informationen über die verwendete App-Version. Da Versionsinformationen für die Sentiment-klassifikation nicht erforderlich sind, wurden diese Reviews im Datensatz beibehalten. Diese sollten nur für Analysen wie (Rating pro Version, Update Impact und Version Comparison) bereinigt werden.

##### Im nächsten Schritt definieren wir die Target Variablen, also Sentiment aus dem Rating.

In [ ]:
def sentiment_label(score):
    if score >= 4:
        return "positive"
    elif score <= 2:
        return "negative"
    else:
        return "neutral"
    
df["sentiment"] = df["score"].apply(sentiment_label)

In [ ]:
#Klassenverteilung prüfen
df["sentiment"].value_counts(normalize=True)

In [ ]:
df.to_csv("../data/BMW/sentiment_clean_reviews.csv")

In [ ]:
df.head()

##### Anschließend machen wir den Train/Test Split und nehmen die Vektoriesierung im Anschluss vor

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["content"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

##### Nun erstellen wir unser erstes Modell. Wir beginnen mit LogisticRegression.

In [ ]:
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

In [ ]:
# sehen wie viele Itertionen gebraucht wurden
print(log_model.n_iter_)

In [ ]:
# Wir erstellen hier die Prediction für das Logistic Regression Modell (Evaluation)
from sklearn.metrics import classification_report
y_pred_log = log_model.predict(X_test)
print(classification_report(y_test, y_pred_log))

In [ ]:
# Hier erstellen wir die Confusion Matrix für das Logistic Regression Modell
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm= confusion_matrix(y_test, y_pred_log, labels=["positive", "neutral", "negative"])
sns.heatmap(cm, annot=True, fmt="d", xticklabels=log_model.classes_, yticklabels=log_model.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

##### Interpretation Confusion Matrix Linear Regression Modell
- Zeilen = echte Labels (True)
- Spalten = Modellvorhersage (Predicted)

- Echte negative Reviews: 952+3+118=1073 --> Davon korrekt erkannt: 952, als positiv klassifiziert = 118, als neutral klassifiziert = 3
- Interpretation: Das Modell erkennt negative Reviews sehr gut, aber manchmal werden negative Reviews als positiv interpretiert.
Das passiert oft bei Sätzen wie: Great app but crashes sometimes.
- Echte neutrale reviews: 59+4+115=178 --> Davon korrekt erkannt: 4
- Interpretation: Das Modell erkennt Neutralität sehr schlecht. Das ist normal bei Sentiment-Modellen, weil neutrale Reviews oft schwer zu unterscheiden sind.
- Echte positive Reviews: 107+2+740=849 -->Davon korrekt erkannt: 740, als negativ klassifizeirt: 107, als neutral klassifiziert: 2
- Interpretation: Das Modell erkennt positive Reviews sehr gut.
- Gesamtinterpretation: Das Modell ist gut bei positive, negative aber schlecht bei neutral. Das ist sehr typisch für Sentimentanalyse. Viele Studien machen deshalb nur negative vs positive.
- Unser Modell ist mit einer accuracy von fast 80% sehr solide.

##### Zum Vergleich testen wir noch das Linear SVM Modell.

In [ ]:
from sklearn.svm import LinearSVC
svm_model = LinearSVC(max_iter=5000)
svm_model.fit(X_train, y_train)


In [ ]:
# sehen wie viele Itertionen gebraucht wurden
print(svm_model.n_iter_)

In [ ]:
# Wir erstellen hier die Prediction für das SVM Modell und den Classification Report (Evaluation)
from sklearn.metrics import classification_report
y_pred_svm= svm_model.predict(X_test)
print(classification_report(y_test, y_pred_svm))


In [ ]:
# Hier erstellen wir die Confusion Matrix für das SVM Modell
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_svm= confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt="d", xticklabels=svm_model.classes_, yticklabels=svm_model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix for SVM")
plt.show()

##### Interpretation Confusion Matrix SVM Modell


In [ ]:
# Hier vergleichen wir die accuracy der beiden Modelle
from sklearn.metrics import accuracy_score
print("Logistic Regression Classification Report:",accuracy_score(y_test, y_pred_log))
print("SVM Classification Report:",accuracy_score(y_test, y_pred_svm))

##### Nachfolgend ermitteln wir mithilfe von GridsearchCV die besten Hyperparameter für unsere Modelle

In [ ]:
#GridSearchCV für Logistic Regression Modell
from sklearn.model_selection import GridSearchCV

param_grid_log= {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}
grid_log= GridSearchCV(LogisticRegression(max_iter=1000,class_weight="balanced"), param_grid_log, cv=5, scoring="accuracy")
grid_log.fit(X_train, y_train)
print("Best Parameters for Logistic Regression:", grid_log.best_params_)
print("Best Score for Logistic Regression:", grid_log.best_score_)

In [ ]:
#Wir evaluieren anschließend das Logistic Regression Modell mit den besten Hyperparametern
best_log_model = grid_log.best_estimator_
y_pred_best_log = best_log_model.predict(X_test)
print(classification_report(y_test, y_pred_best_log))

In [ ]:
#GridSearchCV für SVM Modell
param_grid_svm= {
    "C": [0.01, 0.1, 1, 10]}
grid_svm= GridSearchCV(LinearSVC(class_weight="balanced",max_iter=5000), param_grid_svm, cv=5, scoring="accuracy")
grid_svm.fit(X_train, y_train)
print("Best Parameters for SVM:", grid_svm.best_params_)
print("Best Score for SVM:", grid_svm.best_score_)

In [ ]:
#Wir evaluieren anschließend das SVM Modell mit den besten Hyperparametern
best_svm_model = grid_svm.best_estimator_
y_pred_best_svm = best_svm_model.predict(X_test)
print(classification_report(y_test, y_pred_best_svm))

In [ ]:
#Cross Validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(best_svm_model, X_train, y_train, cv=5)

print("Cross validation scores:", scores)
print("Mean CV score:", scores.mean())

##### Modellvergleich und Auswahl des finalen Modells
Sowohl Logistic Regression als auch Linear SVM erzielten eine sehr ähnliche Leistung mit einer Genauigkeit von etwas 80%. Allerdings schnitt das Linear-SVM-Modell leicht besser ab als die Logistic Regression. Aus diesem Grund wurde Linear SVM als finales Modell für die Sentimentklassifikation ausgewählt.

Nachfolgend schauen wir uns die wichtigsten Wörter im Modell an.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefficients = best_svm_model.coef_
for i, label in enumerate(best_svm_model.classes_):
    top10 = coefficients[i].argsort()[-10:]
    print(f"Top 10 words for {label}:")
    for j in top10:
        print(feature_names[j])

### Wir speichern das Modell

In [ ]:
import joblib

joblib.dump(best_svm_model, "../models/sentiment_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

In [ ]:
#-------------------------------------------------------------------------------------------

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pickle

In [2]:
#Load Clean Data
df = pd.read_csv("../data/BMW/clean_reviews.csv")
print(df.shape)
df.head()

(10150, 4)


,text,rating,date,sentiment
0,every time I tap on the widget to open the app...,3,2026-03-07 16:35:07,neutral
1,I have a vehicle with comfort access and a Sam...,2,2026-03-07 14:42:54,negative
2,Can't add my e34 and e39 :/,1,2026-03-07 09:50:37,negative
3,BMW stands for its reliability n performance i...,5,2026-03-07 07:47:33,positive
4,The lack of support for Octopus Intelligent Go...,2,2026-03-06 19:38:16,negative


In [3]:
#Target definieren
print(df["sentiment"].value_counts())

sentiment
positive    5174
negative    4035
neutral      941
Name: count, dtype: int64


In [4]:
#Train-Test-Split
X = df["text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
#TF-IDF Vektorisierung
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [6]:
#Modelle definieren
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC()
}

In [7]:
#Training & Evaluation
results = {}

for name, model in models.items():
    
    print(f"\n===== {name} =====")
    
    # train
    model.fit(X_train_vec, y_train)
    
    # predict
    y_pred = model.predict(X_test_vec)
    
    # metrics
    acc = accuracy_score(y_test, y_pred)
    
    print("Accuracy:", acc)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    results[name] = acc


===== Logistic Regression =====
Accuracy: 0.8113300492610838

Classification Report:
              precision    recall  f1-score   support

    negative       0.76      0.89      0.82       807
     neutral       0.19      0.02      0.03       188
    positive       0.86      0.90      0.88      1035

    accuracy                           0.81      2030
   macro avg       0.60      0.60      0.58      2030
weighted avg       0.76      0.81      0.78      2030


Confusion Matrix:
[[717   8  82]
 [121   3  64]
 [103   5 927]]

===== Linear SVM =====
Accuracy: 0.7975369458128079

Classification Report:
              precision    recall  f1-score   support

    negative       0.77      0.86      0.81       807
     neutral       0.19      0.06      0.10       188
    positive       0.86      0.88      0.87      1035

    accuracy                           0.80      2030
   macro avg       0.61      0.60      0.59      2030
weighted avg       0.76      0.80      0.77      2030


Confusion

/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


In [8]:
#Bestes Modell wählen
best_model_name = max(results, key=results.get)
print("Best model:", best_model_name)


Best model: Logistic Regression


In [9]:
#Bestes Modell speichern
best_model = models[best_model_name]

# nochmal trainieren auf gesamten Daten
best_model.fit(X_train_vec, y_train)

# speichern
with open("../models/sentiment_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("✅ Model gespeichert!")

✅ Model gespeichert!
